# Strands Agents with Bedrock AgentCore Code Interpreter

This lab demonstrates how to use Amazon Bedrock AgentCore Code Interpreter to give your AI agent the ability to execute Python code dynamically — applied to financial services use cases.

## Overview

In this lab, you will:
- Use the default Code Interpreter to run financial calculations in a sandbox
- Analyze transaction data for fraud patterns
- Calculate portfolio risk metrics (VaR, sector concentration)
- Create a custom Code Interpreter with network access for live market data

## Why Code Interpreter for FSI?

Financial services require:
- **Dynamic calculations** — Risk models, stress tests, scenario analysis
- **Data analysis** — Fraud detection, anomaly identification
- **Secure execution** — Sandboxed environment for sensitive financial data
- **Audit trail** — Every calculation is traceable

## Prerequisites

Ensure you have AWS credentials configured and Nova Pro model access enabled.

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"] = ""
#os.environ["AWS_SECRET_ACCESS_KEY"] = ""
#os.environ["AWS_SESSION_TOKEN"] = ""
#os.environ["AWS_REGION"] = ""

In [ ]:
#%pip install -q strands-agents strands-agents-tools rich bedrock-agentcore pandas

In [1]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Region: {region}")
print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

Region: ap-southeast-2
Nova Pro Model ID: apac.amazon.nova-pro-v1:0


## Part 1: Default Code Interpreter — Financial Calculations

The default Code Interpreter runs Python in a **sandboxed environment** with no network access. Perfect for secure financial calculations.

Let's test it with a portfolio risk calculation:

In [3]:
from strands import Agent
from strands.models import BedrockModel
from strands_tools.code_interpreter import AgentCoreCodeInterpreter

# Initialize the AgentCore Code Interpreter (default: sandboxed, no network)
agentcore_code_interpreter = AgentCoreCodeInterpreter()

# Create agent with default Code Interpreter (sandboxed)
risk_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt="""You are a quantitative analyst assistant. You write and execute Python code 
    to perform financial calculations. Keep responses concise.""",
    tools=[agentcore_code_interpreter.code_interpreter],
)

risk_agent("Calculate the future value of a $2,000,000 investment at 4.8% annual rate compounded monthly after 5 years.")

<thinking> To calculate the future value of the investment, I need to use the formula for compound interest, which is FV = PV * (1 + r/n)^(n*t), where PV is the present value, r is the annual interest rate, n is the number of times interest is compounded per year, and t is the number of years. In this case, PV = 2,000,000, r = 0.048, n = 12 (since the interest is compounded monthly), and t = 5. I can use the code_interpreter tool to execute the Python code that calculates the future value. </thinking>

Tool #1: code_interpreter
The future value of a $2,000,000 investment at 4.8% annual rate compounded monthly after 5 years is approximately $2,541,281.44.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The future value of a $2,000,000 investment at 4.8% annual rate compounded monthly after 5 years is approximately $2,541,281.44.'}], 'metadata': {'usage': {'inputTokens': 4155, 'outputTokens': 47, 'totalTokens': 4202}, 'metrics': {'latencyMs': 804, 'timeToFirstByteMs': 528}}}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'code_interpreter': ToolMetrics(tool={'toolUseId': 'tooluse_qxfEsZD2E0jWtwL1iRjL37', 'name': 'code_interpreter', 'input': {'code_interpreter_input': {'action': {'type': 'executeCode', 'code': 'PV = 2000000\nr = 0.048\nn = 12\nt = 5\nFV = PV * (1 + r/n)**(n*t)\nprint(FV)', 'language': 'python'}}}}, call_count=1, success_count=1, error_count=0, total_time=1.3808550834655762)}, cycle_durations=[3.7175519466400146, 0.8669896125793457], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='50529de5-0e87-490e-a480-cd1f118a5d80', usage={'inputTokens'

## Part 2: Fraud Detection on Transaction Data

Now let's give the agent our synthetic transaction dataset and ask it to identify fraud patterns.

The dataset (`data/transactions.csv`) contains 25 transactions with several suspicious patterns:
- **Velocity attack** — Multiple high-value transactions within seconds
- **Geo-anomaly** — Transactions in different countries within minutes
- **Escalating amounts** — Progressively larger transactions (testing limits)
- **Unusual timing** — High-value transactions at 3am

In [4]:
# Load the transaction data so we can pass it to the agent
import pandas as pd

transactions_df = pd.read_csv("../data/transactions.csv")
print(f"Loaded {len(transactions_df)} transactions")
transactions_df.head()

Loaded 40 transactions


,transaction_id,timestamp,customer_id,amount,currency,merchant,category,location_city,location_country,card_type,is_online
0,TXN-001,2026-05-28 08:15:23,CUST-4421,12.5,AUD,Morning Brew Cafe,Food & Drink,Sydney,AU,debit,False
1,TXN-002,2026-05-28 08:17:45,CUST-4421,3200.0,AUD,TechWorld Electronics,Electronics,Lagos,NG,debit,True
2,TXN-003,2026-05-28 08:18:12,CUST-4421,2800.0,AUD,GiftCards Express,Gift Cards,Lagos,NG,debit,True
3,TXN-004,2026-05-28 08:19:01,CUST-4421,1500.0,AUD,Crypto Exchange XYZ,Financial Services,Moscow,RU,debit,True
4,TXN-005,2026-05-28 12:30:00,CUST-4421,15.8,AUD,Lunch Spot,Food & Drink,Sydney,AU,debit,False


In [6]:
# Pass the data as context and ask the agent to analyze it
transaction_data = transactions_df.to_csv(index=False)

risk_agent(f"""Write Python code to flag fraudulent transactions from this CSV data.

Rules (assign risk_score in parentheses):
- HIGH_AMOUNT (9): amount > 5000
- VELOCITY (6): same customer_id has 3+ rows within 5 minutes
- ODD_HOURS (4): hour between 0 and 5

Use io.StringIO to parse. Print a table: transaction_id, customer_id, amount, flag_reason, risk_score.
If a transaction matches multiple rules, combine them and sum the scores.
Sort by risk_score desc.

Data:
{transaction_data}""")


<thinking> To flag fraudulent transactions, I need to parse the CSV data, apply the given rules to each transaction, and then sort the results by risk_score in descending order. I will use Python's pandas library to handle the data and apply the rules. The rules include checking for HIGH_AMOUNT, VELOCITY, and ODD_HOURS. If a transaction matches multiple rules, I will combine the reasons and sum the scores. Finally, I will print the results in the specified format. </thinking> 
Tool #3: code_interpreter
Here are the flagged transactions sorted by risk_score in descending order:

| transaction_id | customer_id | amount | flag_reason            | risk_score |
|----------------|-------------|--------|------------------------|------------|
| TXN-026        | CUST-6678   | 7500.00| HIGH_AMOUNT ODD_HOURS VELOCITY | 19         |
| TXN-026        | CUST-6678   | 7500.00| HIGH_AMOUNT ODD_HOURS  | 13         |
| TXN-027        | CUST-6678   | 7500.00| HIGH_AMOUNT ODD_HOURS  | 13         |
| TXN-0

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Here are the flagged transactions sorted by risk_score in descending order:\n\n| transaction_id | customer_id | amount | flag_reason            | risk_score |\n|----------------|-------------|--------|------------------------|------------|\n| TXN-026        | CUST-6678   | 7500.00| HIGH_AMOUNT ODD_HOURS VELOCITY | 19         |\n| TXN-026        | CUST-6678   | 7500.00| HIGH_AMOUNT ODD_HOURS  | 13         |\n| TXN-027        | CUST-6678   | 7500.00| HIGH_AMOUNT ODD_HOURS  | 13         |\n| TXN-028        | CUST-6678   | 7500.00| HIGH_AMOUNT ODD_HOURS  | 13         |\n| TXN-009        | CUST-9156   | 9999.99| HIGH_AMOUNT            | 9          |\n| TXN-010        | CUST-9156   | 9999.99| HIGH_AMOUNT            | 9          |\n| TXN-011        | CUST-9156   | 9999.99| HIGH_AMOUNT            | 9          |\n| TXN-012        | CUST-9156   | 9999.99| HIGH_AMOUNT            | 9          |\n| TXN-039      

## Part 3: Portfolio Risk Analysis (VaR)

Let's analyze a portfolio using Value at Risk (VaR) — a standard risk metric in financial services.

We'll use the portfolio data from `data/portfolio.csv`.

In [7]:
portfolio_df = pd.read_csv("../data/portfolio.csv")
print(f"Loaded {len(portfolio_df)} positions")
portfolio_df.head(10)

Loaded 15 positions


,client_id,client_name,asset_class,ticker,units,purchase_price,current_price,currency,weight_pct,sector
0,CLI-001,Acme Super Fund,Equity,CBA.AX,15000,95.2,112.45,AUD,18.5,Financials
1,CLI-001,Acme Super Fund,Equity,BHP.AX,12000,42.8,45.60,AUD,12.2,Materials
2,CLI-001,Acme Super Fund,Equity,CSL.AX,3000,280.0,295.50,AUD,9.8,Healthcare
3,CLI-001,Acme Super Fund,Equity,WBC.AX,20000,22.5,25.80,AUD,8.6,Financials
4,CLI-001,Acme Super Fund,Equity,NAB.AX,18000,28.9,32.10,AUD,7.9,Financials
5,CLI-001,Acme Super Fund,Fixed Income,GOVT.AX,50000,100.0,98.50,AUD,15.0,Government Bonds
6,CLI-001,Acme Super Fund,Fixed Income,IAF.AX,30000,100.0,101.20,AUD,10.5,Corporate Bonds
7,CLI-001,Acme Super Fund,International,VGS.AX,8000,85.0,98.20,AUD,10.8,Global Equity
8,CLI-001,Acme Super Fund,Cash,CASH,500000,1.0,1.00,AUD,6.7,Cash
9,CLI-002,XYZ Treasury,Equity,AAPL,5000,175.0,198.50,USD,22.0,Technology


In [9]:
portfolio_data = portfolio_df.to_csv(index=False)

risk_agent(f"""Analyze this portfolio for risk metrics. Calculate:
1. Total portfolio value (current prices × units) for each client
2. Sector concentration — what % is in each sector? Flag if any sector > 30%
3. Unrealized P&L per position (current vs purchase price)
4. Asset class allocation (Equity vs Fixed Income vs Cash vs Other)

Present results as a clear summary with any risk warnings.

Portfolio data:
{portfolio_data}""")

<thinking> To analyze the portfolio for risk metrics, I need to perform several calculations and assessments. First, I will calculate the total portfolio value for each client by multiplying the current prices by the units for each asset. Then, I will determine the sector concentration by calculating the percentage of the portfolio in each sector and flagging any sector that exceeds 30%. Next, I will calculate the unrealized P&L for each position by comparing the current price to the purchase price. Finally, I will determine the asset class allocation by summing the weights of each asset class. I will present the results in a clear summary with any risk warnings. </thinking> 
Tool #5: code_interpreter
Here is the summary of the portfolio risk metrics:

**Total Portfolio Value:**
- Acme Super Fund (CLI-001): AUD 13,460,850
- XYZ Treasury (CLI-002): USD 17,862,500

**Sector Concentration:**
- Acme Super Fund (CLI-001):
  - Cash: 6.7%
  - Corporate Bonds: 10.5%
  - Financials: 35.0% (Flag

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Here is the summary of the portfolio risk metrics:\n\n**Total Portfolio Value:**\n- Acme Super Fund (CLI-001): AUD 13,460,850\n- XYZ Treasury (CLI-002): USD 17,862,500\n\n**Sector Concentration:**\n- Acme Super Fund (CLI-001):\n  - Cash: 6.7%\n  - Corporate Bonds: 10.5%\n  - Financials: 35.0% (Flagged)\n  - Global Equity: 10.8%\n  - Government Bonds: 15.0%\n  - Healthcare: 9.8%\n  - Materials: 12.2%\n- XYZ Treasury (CLI-002):\n  - Cash: 7.0%\n  - Digital Assets: 12.5%\n  - Government Bonds: 28.0%\n  - Technology: 52.5% (Flagged)\n\n**Unrealized P&L per Position:**\n- CBA.AX: AUD 258,750\n- BHP.AX: AUD 33,600\n- CSL.AX: AUD 46,500\n- WBC.AX: AUD 66,000\n- NAB.AX: AUD 57,600\n- GOVT.AX: AUD -75,000\n- IAF.AX: AUD 36,000\n- VGS.AX: AUD 105,600\n- CASH: AUD 0\n- AAPL: USD 117,500\n- MSFT: USD 135,000\n- AMZN: USD 34,000\n- US10Y: USD -170,000\n- BTC: USD 325,000\n- CASH: USD 0\n\n**Asset Class Allocatio

## Part 4: Custom Code Interpreter with Network Access

The default Code Interpreter is sandboxed (no internet). For use cases that need live data (e.g., fetching real stock prices), we create a **custom Code Interpreter with network access**.

### Step 1: Initialize AgentCore Clients

In [10]:
from bedrock_agentcore._utils import endpoints
import boto3

region = boto3.session.Session().region_name

data_plane_endpoint = endpoints.get_data_plane_endpoint(region)
control_plane_endpoint = endpoints.get_control_plane_endpoint(region)

cp_client = boto3.client('bedrock-agentcore-control',
                        region_name=region,
                        endpoint_url=control_plane_endpoint)

dp_client = boto3.client('bedrock-agentcore',
                        region_name=region,
                        endpoint_url=data_plane_endpoint)

print(f'✅ AgentCore clients initialized (region: {region})')

✅ AgentCore clients initialized (region: ap-southeast-2)


### Step 2: Create Custom Code Interpreter with Network Access

In [11]:
from botocore.exceptions import ClientError

interpreter_name = 'fsi_risk_analyzer'

try:
    interpreter_response = cp_client.create_code_interpreter(
        name=interpreter_name,
        description='FSI Code Interpreter with network access for live market data',
        networkConfiguration={'networkMode': 'PUBLIC'}
    )
    interpreter_id = interpreter_response['codeInterpreterId']
    print(f'✅ Created interpreter: {interpreter_id}')
except ClientError as e:
    if 'already exists' in str(e):
        for item in cp_client.list_code_interpreters()['codeInterpreterSummaries']:
            if item['name'] == interpreter_name:
                interpreter_id = item['codeInterpreterId']
                print(f'✅ Using existing interpreter: {interpreter_id}')
                break
    else:
        raise e

✅ Using existing interpreter: fsi_risk_analyzer-vORoi4eDnY


### Step 3: Create a Session and Test Live Data Access

In [12]:
# Create a session in the custom code interpreter
session_response = dp_client.start_code_interpreter_session(
    codeInterpreterIdentifier=interpreter_id
)
session_id = session_response['sessionId']
print(f'✅ Session created: {session_id}')

# Helper to run code and extract output
def run_code(code, name='executeCode'):
    args = {'code': code, 'language': 'python'} if name == 'executeCode' else {'command': code}
    response = dp_client.invoke_code_interpreter(
        codeInterpreterIdentifier=interpreter_id,
        sessionId=session_id,
        name=name,
        arguments=args
    )
    for event in response.get('stream', []):
        if 'result' in event:
            content = event['result'].get('content', [])
            return '\n'.join(c.get('text', '') for c in content)
    return ''

# Install yfinance
print('Installing yfinance...')
run_code('pip install -q yfinance', name='executeCommand')

# Fetch live stock price
output = run_code("import yfinance as yf\ncba = yf.Ticker('CBA.AX')\ndata = cba.history(period='1d')\nprint(f'CBA.AX Live Price: ${data[\"Close\"].iloc[-1]:.2f} AUD')")
print(output)

✅ Session created: 01KTB0MVB0HAFS0A9Y6CQNY0J2
Installing yfinance...
CBA.AX Live Price: $160.97 AUD


### Step 4: Use Custom Code Interpreter with Strands Agent

In [13]:
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def execute_python(code: str) -> str:
    '''Execute Python code in a secure sandbox with internet access.

    Args:
        code: Python code to execute
    '''
    return run_code(code)

# Create agent with custom code interpreter
live_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='''You are a quantitative analyst with access to live market data.
    Execute Python code to fetch real-time prices using yfinance (already installed).
    Keep responses concise.''',
    tools=[execute_python],
)

live_agent('Fetch the current prices of CBA.AX and WBC.AX and compare their P/E ratios.')

<thinking> To fetch the current prices and compare the P/E ratios of CBA.AX and WBC.AX, I need to execute Python code that uses the yfinance library. First, I will fetch the current prices of both stocks. Then, I will fetch their P/E ratios and compare them. </thinking>

Tool #1: execute_python
The current price of CBA.AX is 160.95 and the current price of WBC.AX is 34.76. The P/E ratio of CBA.AX is 25.92 and the P/E ratio of WBC.AX is 17.12.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The current price of CBA.AX is 160.95 and the current price of WBC.AX is 34.76. The P/E ratio of CBA.AX is 25.92 and the P/E ratio of WBC.AX is 17.12.'}], 'metadata': {'usage': {'inputTokens': 729, 'outputTokens': 66, 'totalTokens': 795}, 'metrics': {'latencyMs': 1120, 'timeToFirstByteMs': 697}}}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'execute_python': ToolMetrics(tool={'toolUseId': 'tooluse_ldwPyryFwye4lQuGSQc6qT', 'name': 'execute_python', 'input': {'code': "import yfinance as yf\n\ncba = yf.Ticker('CBA.AX')\nwbc = yf.Ticker('WBC.AX')\ncba_price = cba.info['regularMarketPrice']\nwbc_price = wbc.info['regularMarketPrice']\ncba_pe = cba.info['trailingPE']\nwbc_pe = wbc.info['trailingPE']\n(cba_price, wbc_price, cba_pe, wbc_pe)"}}, call_count=1, success_count=1, error_count=0, total_time=2.1723361015319824)}, cycle_durations=[3.9624948501586914, 1.1592378616333008], agent_invocations=

## Examining the Agent Loop

In [14]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {live_agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta", max_width=60)
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan", max_width=40)
table.add_column("Tool Result", style="cyan", max_width=40)

for message in live_agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(
        message["role"], (text[-1][:200] + "...") if text and len(text[-1]) > 200 else (text[-1] if text else ""),
        tool_name[-1] if tool_name else "",
        (json.dumps(tool_input[-1])[:150] + "...") if tool_input else "",
        (json.dumps(tool_result[-1])[:150] + "...") if tool_result else ""
    )

console.print(table)

Agent Loop Detail

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Number of Loops: 2

                                                  Agent Messages                                                   
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role      ┃ Text                      ┃ Tool Name      ┃ Tool Input                 ┃ Tool Result               ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ user      │ Fetch the current prices  │                │                            │                           │
│           │ of CBA.AX and WBC.AX and  │                │                            │                           │
│           │ compare their P/E ratios. │                │                            │                           │
├───────────┼───────────────────────────┼────────────────┼────────────────────────────┼───────────────────────────┤
│ assistant │ <thinking> To fetch the   │ execute_python │ {"code": "import yfinance  │                           │
│           │ current prices and        │                │ as yf\n\ncba =             │                           │
│           │ compare the P/E ratios of │                │ yf.Ticker('CBA.AX')\nwbc = │                           │
│           │ CBA.AX and WBC.AX, I need │                │ yf.Ticker('WBC.AX')\ncba_… │                           │
│           │ to execute Python code    │                │ =                          │                           │
│           │ that uses the yfinance    │                │ cba.info['regularMarketPr… │                           │
│           │ library. First, I will    │                │ = wbc.i...                 │                           │
│           │ fetch the current prices  │                │                            │                           │
│           │ of both sto...            │                │                            │                           │
├───────────┼───────────────────────────┼────────────────┼────────────────────────────┼───────────────────────────┤
│ user      │                           │                │                            │ {"text": "(160.95, 34.76, │
│           │                           │                │                            │ 25.917873,                │
│           │                           │                │                            │ 17.123152)"}...           │
├───────────┼───────────────────────────┼────────────────┼────────────────────────────┼───────────────────────────┤
│ assistant │ The current price of      │                │                            │                           │
│           │ CBA.AX is 160.95 and the  │                │                            │                           │
│           │ current price of WBC.AX   │                │                            │                           │
│           │ is 34.76. The P/E ratio   │                │                            │                           │
│           │ of CBA.AX is 25.92 and    │                │                            │                           │
│           │ the P/E ratio of WBC.AX   │                │                            │                           │
│           │ is 17.12.                 │                │                            │                           │
└───────────┴───────────────────────────┴────────────────┴────────────────────────────┴───────────────────────────┘

## Resource Cleanup (Optional)

Clean up the custom Code Interpreter to avoid charges:

In [ ]:
dp_client.stop_code_interpreter_session(
#     codeInterpreterIdentifier=interpreter_id,
#     sessionId=session_id
# )
cp_client.delete_code_interpreter(codeInterpreterId=interpreter_id)
print('✅ Resources cleaned up')

## Summary

In this lab, you:

- ✅ Used the default Code Interpreter for secure financial calculations
- ✅ Analyzed transaction data for fraud patterns (velocity, geo-anomaly, timing)
- ✅ Calculated portfolio risk metrics (sector concentration, P&L, allocation)
- ✅ Created a custom Code Interpreter with network access for live market data
- ✅ Fetched real-time stock prices and compared bank P/E ratios

### FSI Takeaways

| Capability | FSI Application |
|-----------|----------------|
| Sandboxed execution | Secure risk calculations on sensitive data |
| Dynamic code generation | Ad-hoc analysis without pre-built reports |
| Network-enabled interpreter | Live market data, API integrations |
| Audit trail (agent loop) | Compliance — every calculation is traceable |

### Next: Lab 02 — Browser Automation
We'll use AgentCore Browser to monitor regulatory websites (APRA, ASX) and extract live financial data.